# MED-US Class Unlearning - Plan A on Kaggle

Forgetting CIFAR-10 class **6 (frog)** from ResNet-18 by evolutionary search over
layer-wise gradient-free weight edits.

**Before running:** set *Accelerator* to **GPU (T4 or P100)** and *Internet* to **On**,
and attach both input datasets (`medus-class-code`, `medus-class-weights`).

Each step fails loudly rather than degrading quietly - a missing checkpoint or an absent
GPU stops the notebook instead of producing hours of meaningless output.


## 1. Copy the project into the writable working directory

`PROJECT_ROOT` is derived from the source file's location, and every relative path in
every config resolves against it. Running from `/kaggle/input` would make `PROJECT_ROOT`
read-only and the first write to `results/` would fail, so the project is copied first.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

INPUT_CODE    = Path("/kaggle/input/medus-class-code/MEDUS_Class_Unlearning")
INPUT_WEIGHTS = Path("/kaggle/input/medus-class-weights")
PROJECT       = Path("/kaggle/working/MEDUS_Class_Unlearning")

assert INPUT_CODE.is_dir(), (
    f"code dataset not found at {INPUT_CODE}. Attach the 'medus-class-code' dataset. "
    f"Available: {sorted(p.name for p in Path('/kaggle/input').glob('*'))}"
)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(INPUT_CODE, PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

print("project at", PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

## 2. Dependencies

**torch is deliberately not installed.** Kaggle preinstalls a build matched to the
assigned accelerator; `pip install torch` resolves a different one and breaks GPU
access. `requirements.txt` omits it for that reason.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("dependencies installed (torch untouched)")

## 3. Verify CUDA

Asserted, not printed-and-hoped. The search on CPU would take hours and silently produce
the same numbers, so a missing GPU must stop the notebook here.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is not available. Set Accelerator to GPU (T4 or P100) in the notebook "
    "settings and restart the session."
)
print("torch  ", torch.__version__)
print("CUDA   ", torch.version.cuda)
print("GPU    ", torch.cuda.get_device_name(0))
print("memory ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 4. Stage the checkpoints and the class split

W_ref **must** be the `_best_dr` file - selected on `D_r_test` accuracy. The `_best`
checkpoint is selected on full-test accuracy, which for a retain-only reference is both
diluted (1,000 of 10,000 test images are frogs it never trained on) and backwards (an
epoch that classifies more frogs correctly scores higher).

In [ ]:
ckpt_dir  = PROJECT / "results" / "checkpoints"
split_dir = PROJECT / "results" / "splits"
ckpt_dir.mkdir(parents=True, exist_ok=True)
split_dir.mkdir(parents=True, exist_ok=True)

assert INPUT_WEIGHTS.is_dir(), (
    f"weights dataset not found at {INPUT_WEIGHTS}. Attach 'medus-class-weights'."
)

REQUIRED = {
    "cifar10_resnet18_seed42_best.pt":  ckpt_dir,   # W_0
    "class6_frog_reference_best_dr.pt": ckpt_dir,   # W_ref
    "cifar10_class6_frog.json":         split_dir,  # the class split
}

for name, destination in REQUIRED.items():
    source = INPUT_WEIGHTS / name
    assert source.is_file(), (
        f"missing {name} in the weights dataset. Present: "
        f"{sorted(p.name for p in INPUT_WEIGHTS.iterdir())}"
    )
    shutil.copy2(source, destination / name)
    print(f"{name:<42} {source.stat().st_size / 1024**2:>8.1f} MB")

### Confirm W_ref came from a finished training run

`_best_dr.pt` is rewritten every time `D_r_test` improves, so it exists and loads
correctly from epoch 1 onwards. Existence proves nothing about completeness. The
checkpoint records its own epoch and the run's planned total, so the two are compared.

In [ ]:
payload  = torch.load(ckpt_dir / "class6_frog_reference_best_dr.pt",
                      map_location="cpu", weights_only=True)
metadata = payload.get("metadata", {})
metrics  = metadata.get("metrics", {})
epoch    = metadata.get("epoch")
planned  = (metadata.get("training_config") or {}).get("epochs")

print(f"epoch         {epoch} of {planned} planned")
print(f"D_r_test acc  {metrics.get('retain_test_acc'):.4f}")
print(f"D_r_test loss {metrics.get('retain_test_loss'):.4f}")
print(f"D_f_test acc  {metrics.get('forget_test_acc'):.4f}   <- diagnostic; ~0 is CORRECT")
print(f"forget class  {metrics.get('forget_class')}")

assert epoch is not None and planned is not None, "checkpoint carries no epoch metadata"
assert epoch >= planned * 0.9, (
    f"W_ref is from epoch {epoch} of {planned} - this is a LIVE checkpoint from a "
    f"training run that had not finished. Re-export it once training completes."
)
assert metrics.get("forget_class") == 6, "W_ref was trained for a different forget class"
print("\nW_ref is from a finished run.")

## 5. Tests

The suite must be green before anything expensive runs. It covers split exactness, the
specific properties each objective is relied upon for, operator-library composition, and
layer-group isolation.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"],
                        capture_output=True, text=True)
print(result.stdout[-3000:])
assert result.returncode == 0, "tests failed - stopping before the search"

## 6. Objective smoke test

Against the real W_ref. Checks that the identity chromosome scores exactly zero edit
cost, that the objectives respond to real edits, and - the important one - that f3 is
not a restatement of f2.

The `f1` value for the identity chromosome is the **gap the search must close**: the
distance between W_0 and W_ref. Read every candidate's f1 against that, not against 0.

In [ ]:
result = subprocess.run(
    [sys.executable, "experiments/smoke_objectives.py",
     "--config", "search/kaggle_class_frog_main.yaml"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])
if result.stderr:
    print("STDERR:", result.stderr[-2000:])
assert result.returncode == 0, "objective smoke test failed"

## 7. Search smoke test - population 4, one generation

Mechanical, not scientific. Eight evaluations say nothing about selectivity; they say
the loop runs end to end, no objective returns `nan`, the selector found connections on a
class split, and the outputs were written.

It inherits the main config, so a pass here is evidence about the config that will
actually run.

In [ ]:
result = subprocess.run(
    [sys.executable, "experiments/run_plan_a.py",
     "--config", "search/kaggle_class_frog_smoke.yaml"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])
if result.stderr:
    print("STDERR:", result.stderr[-2000:])
assert result.returncode == 0, "smoke search failed - do not run the main search"

## 8. The main Plan A search

Population 10, 50 generations, MicroGA / NSGA-II over the eight safe gradient-free
operators with class-informed selection.

```
f1 = JS( P_ref(D_f) || P_cand(D_f) )     bounded by ln 2
f2 = L_r                                  retain loss
f3 = ||theta - theta_0|| / ||theta_0||    edit cost
```

In [ ]:
result = subprocess.run(
    [sys.executable, "experiments/run_plan_a.py",
     "--config", "search/kaggle_class_frog_main.yaml"],
    capture_output=True, text=True,
)
print(result.stdout[-10000:])
if result.stderr:
    print("STDERR:", result.stderr[-2000:])
assert result.returncode == 0, "main search failed"

## 9. Full-fidelity evaluation

The search screens on subsets. **Nothing above is reportable until it has been
re-measured on the complete sets**, which is what this does - including `D_f_test`, the
1,000 unseen frogs that are the headline result, and the gap to W_ref on every metric.

In [ ]:
result = subprocess.run(
    [sys.executable, "experiments/evaluate_class_front.py",
     "--config", "search/kaggle_class_frog_main.yaml",
     "--front", "results/search/kaggle_class_frog_main/pareto_front.csv"],
    capture_output=True, text=True,
)
print(result.stdout[-10000:])
if result.stderr:
    print("STDERR:", result.stderr[-2000:])
assert result.returncode == 0, "full-fidelity evaluation failed"

## 10. Collect the outputs

Everything under `results/` is written to `/kaggle/working` and can be downloaded from
the notebook's Output tab. The zip excludes checkpoints - they came from the input
dataset and do not need to travel back.

In [ ]:
import zipfile

bundle  = Path("/kaggle/working/plan_a_results.zip")
results = PROJECT / "results"

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in results.rglob("*"):
        if path.is_file() and path.suffix not in (".pt", ".pth"):
            archive.write(path, arcname=str(path.relative_to(PROJECT)))

print(f"wrote {bundle}  ({bundle.stat().st_size / 1024:.1f} KB)")
for name in zipfile.ZipFile(bundle).namelist()[:40]:
    print("  ", name)